In [5]:
import pandas as pd

df = pd.read_csv("question/questions.csv")



In [ ]:
df1 = pd.read_csv("question/results_h.csv")
i = 42
row = df.loc[i]

print(f"""
QUESTION:
{row['question']}

EXPECTED ANSWER:
{row['answer']}


OUTPUT:
{df1.loc[i, 'output']}
""")




QUESTION:
What was Alphabet's customer retention rate for Google Cloud in 2024?

EXPECTED ANSWER:
"The 10-K filing does not disclose customer retention rates for Google Cloud. This type of operational metric is not typically included in SEC annual reports. Revenue growth (31%) and remaining performance obligations can provide indirect indicators of customer commitment."


OUTPUT:
I don't have enough information to answer this question.



In [3]:
from sentence_transformers import SentenceTransformer, util

# Load once and reuse
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def semantic_similarity_bert(text1: str, text2: str) -> float:
    """
    Return semantic similarity between two strings as a float.
    Higher = more semantically similar.
    """
    embeddings = model.encode([text1, text2], convert_to_tensor=True)
    score = util.cos_sim(embeddings[0], embeddings[1])
    return float(score.item())


# Example
s1 = "I love learning machine learning."
s2 = "I enjoy studying AI and data models."

print(semantic_similarity_bert(s1, s2))

c:\Users\hanyi\anaconda3\envs\chatbot-aieb\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3269.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.6332422494888306


In [4]:
import pandas as pd

# correct answers
df = pd.read_csv("question/questions.csv")

letters = list("abcdefghijklmnoprs") + ['base']

results_df1 = pd.DataFrame(index=range(54), columns=letters)

refusal_text = "I don't have enough information to answer this question."

# new dataframe to store refusal counts
refusal_df = pd.DataFrame(index=range(54), columns=letters)

for letter in letters:
    
    df_model = pd.read_csv(f"question/results_{letter}.csv")
    
    scores = []
    refusal_flags = []

    for i in range(54):
        
        level = str(df.loc[i, "level"]).lower()
        output = str(df_model.loc[i, "output"])
        
        # case 1: adversarial question
        if level == "adversarial":
            if refusal_text in output:
                score = 1
            else:
                score = 0
        
        # case 2: non-adversarial but model refused
        elif refusal_text == output:
            score = 0
        
        # case 3: normal similarity evaluation
        else:
            score = semantic_similarity_bert(
                df.loc[i, "answer"],
                output
            )

        scores.append(score)

        # --- NEW CONDITION ---
        if level != "adversarial" and refusal_text in output:
            refusal_flags.append(1)
        else:
            refusal_flags.append(0)

    results_df1[letter] = scores
    refusal_df[letter] = refusal_flags

In [8]:
df_compare1 = pd.DataFrame({    
    "Easy": results_df1[df['level'] == 'Easy'].replace(0, pd.NA).mean(),
    "Advanced": results_df1[df['level'] == 'Advanced'].replace(0, pd.NA).mean(),
    "Challenge": results_df1[df['level'] == 'Challenge'].replace(0, pd.NA).mean(),
    "Financial": results_df1[df['category'] == 'Financial'].replace(0, pd.NA).mean(),
    "Business": results_df1[df['category'] == 'Business'].replace(0, pd.NA).mean(),
    "Risk": results_df1[df['category'] == 'Risk'].replace(0, pd.NA).mean(),
    "Performance": results_df1[df['category'] == 'Performance'].replace(0, pd.NA).mean(),
    "Overall": results_df1[df['level'] != 'Adversarial'].replace(0, pd.NA).mean(),

    # count zeros
    "Adversarial_answered": results_df1[df['level'] == 'Adversarial'].eq(0).sum(),
    "NonAdversarial_not_answered": results_df1[df['level'] != 'Adversarial'].eq(0).sum(),
    "NonAdversarial_partially_answered": refusal_df[df['level'] != 'Adversarial'].sum()-results_df1[df['level'] != 'Adversarial'].eq(0).sum()
})

In [9]:
df_compare1


,Easy,Advanced,Challenge,Financial,Business,Risk,Performance,Overall,Adversarial_answered,NonAdversarial_not_answered,NonAdversarial_partially_answered
a,0.585331,0.637382,NaN,0.523246,0.588294,0.686817,0.763564,0.609826,0,31,0
b,0.564462,0.742738,0.622314,0.624126,0.664579,0.641326,0.750568,0.655882,0,11,2
c,0.570548,0.770986,0.725302,0.647486,0.679602,0.692042,0.802253,0.697296,0,2,2
d,0.593255,0.78401,0.721481,0.686432,0.696874,0.709616,0.790861,0.713295,0,4,2
e,0.556798,0.773675,0.711377,0.640844,0.664554,0.700321,0.807547,0.691291,0,7,0
f,0.553163,0.786903,0.69819,0.651461,0.683579,0.703751,0.795679,0.695959,0,2,1
g,0.583036,0.747932,0.636955,0.649619,0.654138,0.646283,0.756797,0.669978,0,12,1
h,0.570653,0.778898,0.802257,0.642306,0.688411,0.709298,0.791242,0.700679,0,6,0
i,0.584269,0.783242,0.75278,0.672792,0.689387,0.699644,0.814029,0.713333,0,4,2
j,0.579673,0.793794,0.652052,0.668624,0.689206,0.712412,0.807218,0.707528,0,5,0


In [24]:
df_compare2 = pd.DataFrame({    
    "Easy": results_df1[df['level'] == 'Easy'].mean(),
    "Advanced": results_df1[df['level'] == 'Advanced'].mean(),
    "Challenge": results_df1[df['level'] == 'Challenge'].mean(),
    "Financial": results_df1[df['category'] == 'Financial'].mean(),
    "Business": results_df1[df['category'] == 'Business'].mean(),
    "Risk": results_df1[df['category'] == 'Risk'].mean(),
    "Performance": results_df1[df['category'] == 'Performance'].mean(),
    "Overall": results_df1[df['level'] != 'Adversarial'].mean(),

    # count zeros
    "Adversarial_answered": results_df1[df['level'] == 'Adversarial'].eq(0).sum(),
    "NonAdversarial_not_answered": results_df1[df['level'] != 'Adversarial'].eq(0).sum(),
    "NonAdversarial_partially_answered": refusal_df[df['level'] != 'Adversarial'].sum()-results_df1[df['level'] != 'Adversarial'].eq(0).sum()
})

In [25]:
df_compare2

,Easy,Advanced,Challenge,Financial,Business,Risk,Performance,Overall,Adversarial_answered,NonAdversarial_not_answered,NonAdversarial_partially_answered
a,0.329249,0.196118,0.000000,0.261623,0.196098,0.257556,0.286336,0.215980,0,31,0
b,0.564462,0.514203,0.311157,0.534965,0.609197,0.480995,0.469105,0.505575,0,11,2
c,0.570548,0.770986,0.483535,0.647486,0.679602,0.692042,0.802253,0.668242,0,2,2
d,0.556177,0.753855,0.480987,0.637402,0.696874,0.709616,0.692003,0.653854,0,4,2
e,0.521999,0.714162,0.237126,0.549295,0.664554,0.700321,0.706604,0.590478,0,7,0
f,0.553163,0.756637,0.581825,0.651461,0.683579,0.703751,0.696219,0.666961,0,2,1
g,0.546597,0.517799,0.318477,0.510415,0.490603,0.484712,0.662197,0.502483,0,12,1
h,0.570653,0.718983,0.267419,0.596427,0.688411,0.709298,0.692337,0.613094,0,6,0
i,0.547752,0.783242,0.376390,0.624735,0.689387,0.699644,0.814029,0.653889,0,4,2
j,0.579673,0.763263,0.217351,0.620865,0.689206,0.712412,0.807218,0.633828,0,5,0


In [ ]:
df_judge = pd.read_csv("llm_evaluation_results_6_models.csv")

In [30]:
hall_df = df_judge.pivot(
    index="question",
    columns="model",
    values="hallucination"
)


In [31]:
acc_df = df_judge.pivot(
    index="question",
    columns="model",
    values="accuracy"
)

In [32]:
rlvc_df = df_judge.pivot(
    index="question",
    columns="model",
    values="relevance"
)

In [33]:
rtq_df = df_judge.pivot(
    index="question",
    columns="model",
    values="retreival quality"
)

In [34]:
df_temp = hall_df

df_compare_hall = pd.DataFrame({
    "Easy": df_temp[df['level'] == 'Easy'].mean(),
    "Advanced": df_temp[df['level'] == 'Advanced'].mean(),
    "Challenge": df_temp[df['level'] == 'Challenge'].mean(),
    'Adversarial':df_temp[df['level'] == 'Adversarial'].mean(),
    "Financial": df_temp[df['category'] == 'Financial'].mean(),
    "Business": df_temp[df['category'] == 'Business'].mean(),
    "Risk": df_temp[df['category'] == 'Risk'].mean(),
    "Performance": df_temp[df['category'] == 'Performance'].mean(),
    "Overall": df_temp.mean()
})

print("hallucination")
df_compare_hall

hallucination


,Easy,Advanced,Challenge,Adversarial,Financial,Business,Risk,Performance,Overall
model,,,,,,,,,
base,4.1250,4.461538,4.666667,1.0,4.142857,4.250000,4.500,4.625,4.000000
d,4.1250,4.038462,4.500000,1.0,4.142857,4.000000,4.000,4.125,3.777778
i,4.0625,4.038462,4.666667,1.0,4.071429,4.083333,4.000,4.000,3.777778
l,4.0000,4.192308,4.833333,1.0,4.142857,4.000000,4.250,4.125,3.851852
n,4.0625,4.000000,4.333333,1.0,4.000000,4.083333,4.000,4.000,3.722222
r,4.0000,4.076923,4.333333,1.0,4.071429,4.000000,4.125,4.000,3.740741


In [35]:
df_temp = acc_df

df_compare_acc = pd.DataFrame({
    "Easy": df_temp[df['level'] == 'Easy'].mean(),
    "Advanced": df_temp[df['level'] == 'Advanced'].mean(),
    "Challenge": df_temp[df['level'] == 'Challenge'].mean(),
    'Adversarial':df_temp[df['level'] == 'Adversarial'].mean(),
    "Financial": df_temp[df['category'] == 'Financial'].mean(),
    "Business": df_temp[df['category'] == 'Business'].mean(),
    "Risk": df_temp[df['category'] == 'Risk'].mean(),
    "Performance": df_temp[df['category'] == 'Performance'].mean(),
    "Overall": df_temp.mean()
})

df_compare_acc

,Easy,Advanced,Challenge,Adversarial,Financial,Business,Risk,Performance,Overall
model,,,,,,,,,
base,3.50,2.153846,1.333333,1.0,3.428571,3.000000,2.0,1.5,2.333333
d,3.50,3.846154,2.000000,1.0,3.428571,4.000000,4.0,3.5,3.222222
i,3.75,3.846154,1.333333,1.0,3.714286,3.666667,4.0,4.0,3.222222
l,4.00,3.230769,0.666667,1.0,3.428571,4.000000,3.0,3.5,2.925926
n,3.75,4.000000,2.666667,1.0,4.000000,3.666667,4.0,4.0,3.444444
r,4.00,3.692308,2.666667,1.0,3.714286,4.000000,3.5,4.0,3.370370


In [36]:
df_temp = rlvc_df

df_compare_rlvc = pd.DataFrame({
    "Easy": df_temp[df['level'] == 'Easy'].mean(),
    "Advanced": df_temp[df['level'] == 'Advanced'].mean(),
    "Challenge": df_temp[df['level'] == 'Challenge'].mean(),
    'Adversarial':df_temp[df['level'] == 'Adversarial'].mean(),
    "Financial": df_temp[df['category'] == 'Financial'].mean(),
    "Business": df_temp[df['category'] == 'Business'].mean(),
    "Risk": df_temp[df['category'] == 'Risk'].mean(),
    "Performance": df_temp[df['category'] == 'Performance'].mean(),
    "Overall": df_temp.mean()
})

df_compare_rlvc

,Easy,Advanced,Challenge,Adversarial,Financial,Business,Risk,Performance,Overall
model,,,,,,,,,
base,4.50,3.153846,2.333333,1.0,4.428571,4.000000,3.0,2.5,3.222222
d,4.50,4.846154,3.000000,1.0,4.428571,5.000000,5.0,4.5,4.111111
i,4.75,4.846154,2.333333,1.0,4.714286,4.666667,5.0,5.0,4.111111
l,5.00,4.230769,1.666667,1.0,4.428571,5.000000,4.0,4.5,3.814815
n,4.75,5.000000,3.666667,1.0,5.000000,4.666667,5.0,5.0,4.333333
r,5.00,4.692308,3.666667,1.0,4.714286,5.000000,4.5,5.0,4.259259


In [37]:
df_temp = rtq_df

df_compare_rtq = pd.DataFrame({
    "Easy": df_temp[df['level'] == 'Easy'].mean(),
    "Advanced": df_temp[df['level'] == 'Advanced'].mean(),
    "Challenge": df_temp[df['level'] == 'Challenge'].mean(),
    'Adversarial':df_temp[df['level'] == 'Adversarial'].mean(),
    "Financial": df_temp[df['category'] == 'Financial'].mean(),
    "Business": df_temp[df['category'] == 'Business'].mean(),
    "Risk": df_temp[df['category'] == 'Risk'].mean(),
    "Performance": df_temp[df['category'] == 'Performance'].mean(),
    "Overall": df_temp.mean()
})

df_compare_rtq

,Easy,Advanced,Challenge,Adversarial,Financial,Business,Risk,Performance,Overall
model,,,,,,,,,
base,1.0000,1.000000,1.0,1.0,1.000000,1.00,1.000,1.000,1.000000
d,3.6250,3.884615,2.5,1.0,3.571429,4.00,4.000,3.625,3.333333
i,3.8125,3.884615,2.0,1.0,3.785714,3.75,4.000,4.000,3.333333
l,4.0000,3.423077,1.5,1.0,3.571429,4.00,3.250,3.625,3.111111
n,3.8125,4.000000,3.0,1.0,4.000000,3.75,4.000,4.000,3.500000
r,4.0000,3.769231,3.0,1.0,3.785714,4.00,3.625,4.000,3.444444


In [ ]:
df_f = pd.read_csv("question/results_n.csv")


In [17]:
df_f[df_f['output'].str.contains("I don't have enough information to answer this question.", na=False)]

,output,citations
18,I don't have enough information to answer this...,[]
42,I don't have enough information to answer this...,[]
43,I don't have enough information to answer this...,[]
44,I don't have enough information to answer this...,[]
45,I don't have enough information to answer this...,[]
46,I don't have enough information to answer this...,[]
47,I don't have enough information to answer this...,[]
49,I don't have enough information to answer this...,[]
53,I don't have enough information to answer this...,"[{""idx"": 1, ""source"": ""Amazon 10K 2024.pdf"", ""..."


In [15]:
df[df['level']== 'Adversarial']

,question,company,category,level,location,answer
42,What was Alphabet's customer retention rate fo...,Alphabet,Adversarial,Adversarial,NaN,"""The 10-K filing does not disclose customer re..."
43,How much did Amazon spend on marketing for Pri...,Amazon,Adversarial,Adversarial,NaN,"""The specific marketing or content budget for ..."
44,What is Microsoft's market share in the cloud ...,Microsoft,Adversarial,Adversarial,NaN,"""The 10-K does not include specific market sha..."
45,What was the closing stock price of Amazon on ...,Amazon,Adversarial,Adversarial,NaN,"""The provided 10-K document does not contain t..."
46,What is the total compensation package for Sat...,Microsoft,Adversarial,Adversarial,NaN,"""The specific compensation details for Satya N..."
47,What did Jeff Bezos say in his recent intervie...,Amazon,Adversarial,Adversarial,NaN,The provided sources do not contain any inform...


In [38]:
df_compare1.to_csv('compare/BERT_fail_not_included.csv')
df_compare2.to_csv('compare/BERT_fail_included.csv')
df_compare_hall.to_csv('compare/LLM_hall.csv')
df_compare_acc.to_csv('compare/LLM_acc.csv')
df_compare_rlvc.to_csv('compare/LLM_rlvc.csv')
df_compare_rtq.to_csv('compare/LLM_rtq.csv')